In [1]:
from cimloader.databases import ConnectionParameters
from cimloader.databases import Neo4jConnection
from cimloader.uploaders import Neo4jUploader

In [2]:
# Neo4J Connection
params = ConnectionParameters(url = "neo4j://localhost:7687", database="neo4j",
                               username="neo4j", password="test1234",
                               container = "gridappsd-docker-neo4j-apoc-1")
neo4j = Neo4jConnection(params)
loader = Neo4jUploader(params)

In [3]:
neo4j.connect()
# neo4j.drop_all()
# neo4j.configure()

In [4]:
# loader.upload_from_file(filepath="./test_models/", filename="ieee13_seto.xml")

In [5]:
from cimgraph.queries import cypher
import cimgraph.data_profile.cimhub_2023 as cim

In [6]:
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')

query = cypher.get_all_nodes_from_container(feeder)
(query)


'MATCH (container:Feeder)\nWHERE container.uri = "urn:uuid:49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"\nMATCH (eq) - [:`Equipment.EquipmentContainer`] - (container)\nOPTIONAL MATCH (cnode) - [:`Terminal.ConnectivityNode`] - (term:Terminal) - [:`Terminal.ConductingEquipment`] -> (eq)\nRETURN DISTINCT\nREPLACE(cnode.uri, "urn:uuid:", "") as ConnectivityNode,\nREPLACE(term.uri, "urn:uuid:", "") as Terminal,\nREPLACE(eq.uri, "urn:uuid:", "") as eq_id,\nLABELS(eq)[1] as eq_class'

In [7]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_URL'] = 'neo4j://localhost:7687'
os.environ['CIMG_DATABASE'] = 'neo4j'
os.environ['CIMG_HOST'] = 'localhost'
os.environ['CIMG_PORT'] = '7687'
os.environ['CIMG_USERNAME'] = 'neo4j'
os.environ['CIMG_PASSWORD'] = 'test1234'
os.environ['CIMG_NAMESPACE'] = 'http://iec.ch/TC57/CIM100#'
os.environ['CIMG_IEC61970_301'] = '8'
os.environ['CIMG_USE_UNITS'] = 'False'

In [8]:
from cimgraph.models import FeederModel
from cimgraph.databases import Neo4jConnection
database = Neo4jConnection()
# feeder = cim.Feeder(mRID=self.feeder_mrid)
network = FeederModel(connection=database, container=feeder, distributed=False)
initial_keys = len(network.graph.keys())
initial_keys

13

In [9]:
print(network.get_edges_query(cim.ACLineSegment))


MATCH (n:ACLineSegment)
WHERE n.uri IN ["urn:uuid:ABF877D7-DAC2-4BF0-AB58-9A8A02E92EB3", 
"urn:uuid:4C04F838-62AA-475E-AEFA-A63B7C889C13", 
"urn:uuid:D34B0D01-B082-4081-A3CC-B68B9B8313A4", 
"urn:uuid:45395C84-F20A-4F5A-977F-B80348256421", 
"urn:uuid:A04CDFB1-E951-4FC4-8882-0323CD70AE3C", 
"urn:uuid:8E180773-2A9B-4136-BC9A-132A52C8E276", 
"urn:uuid:FBE26B35-13AB-457D-9795-DF58B28E309D", 
"urn:uuid:ABF53597-A808-422A-B7EE-552F24D83A5F", 
"urn:uuid:7060D0BB-B30D-4932-8FA1-40820A0FC4D0", 
"urn:uuid:0BBD0EA3-F665-465B-86FD-FC8B8466AD53", 
"urn:uuid:1C6781A2-5B9D-4525-8A9B-F9B32C4C4AC0"]
OPTIONAL MATCH (n) - [r] - (m)
RETURN DISTINCT n.uri as identifier,
type(r) as attribute,
REPLACE(m.uri, "urn:uuid:", "") as edge_id,
labels(m)[1] as edge_class


In [10]:
from uuid import UUID
network.get_all_edges(cim.ACLineSegment)
line = network.graph[cim.ACLineSegment][UUID('0bbd0ea3-f665-465b-86fd-fc8b8466ad53')]
line.pprint()

{
    "@id": "0bbd0ea3-f665-465b-86fd-fc8b8466ad53",
    "@type": "ACLineSegment",
    "name": "645646",
    "Location": {
        "@id": "89c21411-d107-454a-93b0-0905893dd27d",
        "@type": "Location"
    },
    "Measurements": [
        {
            "@id": "b5d54a83-8b4f-4675-9696-1a6d8ced1d1d",
            "@type": "Analog"
        },
        {
            "@id": "f63a8e39-4f96-475a-83c2-f1adf5441159",
            "@type": "Analog"
        },
        {
            "@id": "b019274b-13e5-4c02-ab6d-471979c70d12",
            "@type": "Analog"
        },
        {
            "@id": "d70108e9-7c81-4aa8-9c1d-4a31b5b0cd2b",
            "@type": "Analog"
        }
    ],
    "EquipmentContainer": {
        "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
        "@type": "Feeder"
    },
    "SubSchedulingArea": {
        "@id": "2cff1e26-8f8f-42b5-b054-87f28523d303",
        "@type": "SwitchArea"
    },
    "BaseVoltage": {
        "@id": "2a158e0c-cd01-4a50-aeba-59d761fcf15d",
        

In [11]:
for cls in network.graph:
    print(cls.__name__)

ACLineSegment
ConnectivityNode
Terminal
EnergyConsumer
TransformerTank
PowerElectronicsConnection
PowerTransformer
LinearShuntCompensator
EnergySource
Fuse
LoadBreakSwitch
Recloser
Breaker
Feeder
Analog
Location
PerLengthPhaseImpedance
BaseVoltage
ACLineSegmentPhase
SwitchArea


In [12]:
# loader.upload_from_url(url="https://raw.githubusercontent.com/GRIDAPPSD/Powergrid-Models/refs/heads/develop/models/feeders/CIM/XML/IEEE13/IEEE13.xml")

# loader.upload_from_url(url="https://raw.githubusercontent.com/GRIDAPPSD/Powergrid-Models/feature/SETO/platform/cimxml/IEEE13_Assets.xml")